In [34]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)


In [35]:


# project_root = Path.cwd().resolve()
# while not (project_root / "data").exists() and project_root != project_root.parent:
#     project_root = project_root.parent

# bronze_path = project_root / "data" / "bronze" / "bronze_liste_communes_france.csv"
# silver_path = project_root / "data" / "silver" / "silver_liste_communes_france.csv"
# silver_path.parent.mkdir(parents=True, exist_ok=True)  # ensure /data/silver exists

# # Read bronze
# df_bronze = pd.read_csv(bronze_path, sep=";", dtype=str, encoding="utf-8")

# # Transform -> silver dataframe
# df_silver = df_bronze.copy()

# # ... your cleaning / renaming / casting on df_silver ...

# # Save silver
# df_silver.to_csv(silver_path, sep=";", index=False, encoding="utf-8")
# print(f"Wrote silver: {silver_path}")


In [36]:
# df_silver.dtypes

In [37]:
# string_cols = [
#     "code_insee","nom_standard","nom_sans_pronom","nom_a","nom_de","nom_sans_accent",
#     "nom_standard_majuscule","typecom","typecom_texte","reg_code","reg_nom","dep_code",
#     "dep_nom","canton_code","canton_nom","epci_code","epci_nom","academie_code",
#     "academie_nom","code_postal","codes_postaux","zone_emploi","code_insee_centre_zone_emploi",
#     "code_unite_urbaine","nom_unite_urbaine","type_commune_unite_urbaine",
#     "statut_commune_unite_urbaine","grille_densite_texte","niveau_equipements_services_texte",
#     "gentile","url_wikipedia","url_villedereve","extraction_source_url","source_file_name"
# ]

# int_cols = [
#     "taille_unite_urbaine","population","superficie_hectare","superficie_km2","densite",
#     "altitude_moyenne","altitude_minimale","altitude_maximale","grille_densite",
#     "niveau_equipements_services"
# ]

# float_cols = ["latitude_mairie","longitude_mairie","latitude_centre","longitude_centre"]

# for c in string_cols:
#     if c in df_silver.columns:
#         df_silver[c] = df_silver[c].str.strip().astype("string")

# for c in int_cols:
#     if c in df_silver.columns:
#         df_silver[c] = pd.to_numeric(df_silver[c], errors="coerce").astype("Int64")

# for c in float_cols:
#     if c in df_silver.columns:
#         df_silver[c] = pd.to_numeric(df_silver[c].str.replace(",", ".", regex=False), errors="coerce").astype("Float64")

# if "ingestion_timestamp" in df_silver.columns:
#     df_silver["ingestion_timestamp"] = pd.to_datetime(df_silver["ingestion_timestamp"], errors="coerce")

# print(df_silver.dtypes)


In [38]:
# print(df_silver.columns.tolist())

In [39]:
# # Save silver
# df_silver.to_csv(silver_path, sep=";", index=False, encoding="utf-8")
# print(f"Wrote silver: {silver_path}")

In [40]:

project_root = Path.cwd().resolve()
while not (project_root / "data").exists() and project_root != project_root.parent:
    project_root = project_root.parent

bronze_path = project_root / "data" / "bronze" / "bronze_liste_communes_france.csv"
silver_path = project_root / "data" / "silver" / "silver_liste_communes_france.csv"
silver_path.parent.mkdir(parents=True, exist_ok=True)

# Read bronze
df_bronze = pd.read_csv(
    bronze_path,
    sep=";",
    dtype=str,
    encoding="utf-8"
)

# Normalize column names
df_bronze.columns = [c.strip() for c in df_bronze.columns]

# Start silver
df_silver = df_bronze.copy()



In [ ]:
# Drop useless technical column 
if "Unnamed: 0" in df_silver.columns:
    df_silver = df_silver.drop(columns=["Unnamed: 0"])

 
# Keep useful columns only
 
cols_keep = [
    "code_insee",
    "nom_standard",
    "typecom",
    "typecom_texte",
    "reg_code",
    "reg_nom",
    "dep_code",
    "dep_nom",
    "canton_code",
    "canton_nom",
    "epci_code",
    "epci_nom",
    "academie_code",
    "academie_nom",
    "code_postal",
    "codes_postaux",
    "zone_emploi",
    "code_insee_centre_zone_emploi",
    "code_unite_urbaine",
    "nom_unite_urbaine",
    "taille_unite_urbaine",
    "type_commune_unite_urbaine",
    "statut_commune_unite_urbaine",
    "population",
    "superficie_hectare",
    "superficie_km2",
    "densite",
    "altitude_moyenne",
    "altitude_minimale",
    "altitude_maximale",
    "latitude_mairie",
    "longitude_mairie",
    "latitude_centre",
    "longitude_centre",
    "grille_densite",
    "grille_densite_texte",
    "niveau_equipements_services",
    "niveau_equipements_services_texte",
    "gentile",
    "extraction_source_url",
    "ingestion_timestamp",
    "source_file_name",
]

existing_cols = [c for c in cols_keep if c in df_silver.columns]
df_silver = df_silver[existing_cols].copy()

 
# Helpers
 
def clean_string(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip()

def to_int_nullable(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").astype("Int64")

def to_float_nullable(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype("string").str.replace(",", ".", regex=False),
        errors="coerce"
    ).astype("Float64")

 
# Cast strings
 
string_cols = [
    "code_insee",
    "nom_standard",
    "typecom",
    "typecom_texte",
    "reg_code",
    "reg_nom",
    "dep_code",
    "dep_nom",
    "canton_code",
    "canton_nom",
    "epci_code",
    "epci_nom",
    "academie_code",
    "academie_nom",
    "code_postal",
    "codes_postaux",
    "zone_emploi",
    "code_insee_centre_zone_emploi",
    "code_unite_urbaine",
    "nom_unite_urbaine",
    "type_commune_unite_urbaine",
    "statut_commune_unite_urbaine",
    "grille_densite_texte",
    "niveau_equipements_services_texte",
    "gentile",
    "extraction_source_url",
    "source_file_name",
]

for col in string_cols:
    if col in df_silver.columns:
        df_silver[col] = clean_string(df_silver[col])

# Preserve leading zeros explicitly on code fields
code_cols = [
    "code_insee",
    "reg_code",
    "dep_code",
    "canton_code",
    "epci_code",
    "academie_code",
    "code_postal",
    "zone_emploi",
    "code_insee_centre_zone_emploi",
    "code_unite_urbaine",
]
for col in code_cols:
    if col in df_silver.columns:
        df_silver[col] = clean_string(df_silver[col])

 
# Cast integers
 
int_cols = [
    "taille_unite_urbaine",
    "population",
    "superficie_hectare",
    "superficie_km2",
    "densite",
    "altitude_moyenne",
    "altitude_minimale",
    "altitude_maximale",
    "grille_densite",
    "niveau_equipements_services",
]

for col in int_cols:
    if col in df_silver.columns:
        df_silver[col] = to_int_nullable(df_silver[col])

 
# Cast floats
 
float_cols = [
    "latitude_mairie",
    "longitude_mairie",
    "latitude_centre",
    "longitude_centre",
]

for col in float_cols:
    if col in df_silver.columns:
        df_silver[col] = to_float_nullable(df_silver[col])

 
# Cast timestamp
 
if "ingestion_timestamp" in df_silver.columns:
    df_silver["ingestion_timestamp"] = pd.to_datetime(
        df_silver["ingestion_timestamp"],
        errors="coerce"
    )

 
# Save silver
 
df_silver.to_csv(silver_path, sep=";", index=False, encoding="utf-8")

print(f"Wrote silver: {silver_path}")
print(f"Shape: {df_silver.shape}")
print("\nDtypes:")
print(df_silver.dtypes)
print("\nSample:")
display(df_silver.head())

Wrote silver: /Users/zainfrayha/Documents/EPSI/MSPR/electio-analytics-poc/data/silver/silver_liste_communes_france.csv
Shape: (266, 42)

Dtypes:
code_insee                                   string
nom_standard                                 string
typecom                                      string
typecom_texte                                string
reg_code                                     string
reg_nom                                      string
dep_code                                     string
dep_nom                                      string
canton_code                                  string
canton_nom                                   string
epci_code                                    string
epci_nom                                     string
academie_code                                string
academie_nom                                 string
code_postal                                  string
codes_postaux                                string
zone_emploi            

,code_insee,nom_standard,typecom,typecom_texte,reg_code,reg_nom,dep_code,dep_nom,canton_code,canton_nom,epci_code,epci_nom,academie_code,academie_nom,code_postal,codes_postaux,zone_emploi,code_insee_centre_zone_emploi,code_unite_urbaine,nom_unite_urbaine,taille_unite_urbaine,type_commune_unite_urbaine,statut_commune_unite_urbaine,population,superficie_hectare,superficie_km2,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude_mairie,longitude_mairie,latitude_centre,longitude_centre,grille_densite,grille_densite_texte,niveau_equipements_services,niveau_equipements_services_texte,gentile,extraction_source_url,ingestion_timestamp,source_file_name
0,69001,Affoux,COM,commune,84,Auvergne-Rhône-Alpes,69,Rhône,6910,Tarare,200040566,CA de l'Ouest Rhodanien,10,Lyon,69170,69170,08430,69243,69000,<NA>,0,HORS UNITE URBAINE,H,395,1068,11,37,750,498,960,45.845,4.404,45.844,4.414,6,Rural à habitat dispersé,0,communes non pôle,Affousiens,https://www.data.gouv.fr/api/1/datasets/r/f5df...,2026-03-07 20:49:48.630460,liste_communes_france.csv
1,69002,Aigueperse,COM,commune,84,Auvergne-Rhône-Alpes,69,Rhône,6911,Thizy-les-Bourgs,200067817,CC Saône-Beaujolais,10,Lyon,69790,69790,08434,69264,69000,<NA>,0,HORS UNITE URBAINE,H,243,1290,13,19,489,403,610,46.277,4.435,46.28,4.423,7,Rural à habitat très dispersé,0,communes non pôle,Aiguepersirons,https://www.data.gouv.fr/api/1/datasets/r/f5df...,2026-03-07 20:49:48.630460,liste_communes_france.csv
2,69003,Albigny-sur-Saône,COM,commune,84,Auvergne-Rhône-Alpes,69,Rhône,6999,Lyon,200046977,Métropole de Lyon,10,Lyon,69250,69250,08421,69123,00760,Lyon,7,UNITE URBAINE,B,2991,266,3,1124,223,167,0,45.871,4.832,45.864,4.831,4,Ceintures urbaines,2,centres intermédiaires d'équipements et de ser...,Albignolais,https://www.data.gouv.fr/api/1/datasets/r/f5df...,2026-03-07 20:49:48.630460,liste_communes_france.csv
3,69004,Alix,COM,commune,84,Auvergne-Rhône-Alpes,69,Rhône,6904,Val d'Oingt,200040574,CC Beaujolais Pierres Dorées,10,Lyon,69380,69380,08421,69123,69301,Val d'Oingt,3,UNITE URBAINE,B,766,360,4,213,310,258,408,45.911,4.653,45.917,4.655,6,Rural à habitat dispersé,0,communes non pôle,Alixois,https://www.data.gouv.fr/api/1/datasets/r/f5df...,2026-03-07 20:49:48.630460,liste_communes_france.csv
4,69005,Ambérieux,COM,commune,84,Auvergne-Rhône-Alpes,69,Rhône,6901,Anse,200040574,CC Beaujolais Pierres Dorées,10,Lyon,69480,69480,08421,69123,00760,Lyon,7,UNITE URBAINE,B,607,461,5,132,171,167,176,45.928,4.737,45.929,4.741,4,Ceintures urbaines,1,centres locaux d'équipements et de services,"Ambarrois, Ambarroises",https://www.data.gouv.fr/api/1/datasets/r/f5df...,2026-03-07 20:49:48.630460,liste_communes_france.csv
